# FIXED COUPON BOND EXAMPLE TREASURY

Question on StackExchange - THIS HAS NOT BEEN REPLICATED AS EXAMPLE LOOKS WRONG

https://quant.stackexchange.com/questions/66508/schedule-yield-to-maturity-and-npv-of-fixed-rate-bond-from-quantlib-python

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils import *
from financepy.products.bonds.bond import *
from financepy.market.curves import *

# Define the Bond

In [3]:
issue_dt = Date(28, 9, 2019)
maturity_dt = Date(28, 9, 2024)
coupon = 0.05
freq_type = FrequencyTypes.SEMI_ANNUAL
day_count_type = DayCountTypes.ACT_360
face = 100.0
us_cal = CalendarTypes.UNITED_STATES

In [4]:
bond = Bond(issue_dt, maturity_dt, coupon, freq_type, day_count_type, cal_type=us_cal)

You can get information about the bond using the print method.

In [5]:
print(bond)

OBJECT TYPE: Bond
ISSUE DATE: 28-SEP-2019
MATURITY DATE: 28-SEP-2024
COUPON (%): 5.0
FREQUENCY: FrequencyTypes.SEMI_ANNUAL
ACCRUAL DC TYPE: DayCountTypes.ACT_360
EX-DIVIDEND DAYS: 0
CALENDAR TYPE: CalendarTypes.UNITED_STATES
BUS DAYS ADJUST: BusDayAdjustTypes.FOLLOWING
DATE GEN RULE: DateGenRuleTypes.BACKWARD
COUPON TYPE: CouponType.FIXED


## Bond Cash Flows

We first need to set the settlement date of the bond. 

In [6]:
value_dt = Date(31, 7, 2020)
settle_dt = value_dt.add_days(100)
print(settle_dt)

08-NOV-2020


In [7]:
bond.accrued_interest(value_dt)

1.7361111111111112

In [8]:
bond.accrued_interest(settle_dt)

0.5694444444444444

In [9]:
print(bond.print_payments(settle_dt, 100))

Coupon Date 	 Payment Date 	         Status          Amount
08-NOV-2020 	             	     SETTLEMENT 
28-MAR-2021 	 29-MAR-2021 	   HOLIDAY ROLL 	            2.50 
28-SEP-2021 	 28-SEP-2021 	      UNCHANGED 	            2.50 
28-MAR-2022 	 28-MAR-2022 	      UNCHANGED 	            2.50 
28-SEP-2022 	 28-SEP-2022 	      UNCHANGED 	            2.50 
28-MAR-2023 	 28-MAR-2023 	      UNCHANGED 	            2.50 
28-SEP-2023 	 28-SEP-2023 	      UNCHANGED 	            2.50 
28-MAR-2024 	 28-MAR-2024 	      UNCHANGED 	            2.50 
28-SEP-2024 	 30-SEP-2024 	       MATURITY 	          102.50 

None


The convention is to use these dates for yield calculations even if some fall on weekends.

In [10]:
spot_dts = [Date(31, 7, 2020), Date(1, 1, 2027)]
spot_rates = [0.01, 0.02]

In [11]:
zero_curve = DiscountCurveZeros(value_dt,
                                spot_dts,
                                spot_rates,
                                freq_type,
                                InterpTypes.LINEAR_ZERO_RATES,
                                DayCountTypes.ACT_360)

In [12]:
print(zero_curve)

OBJECT TYPE: DiscountCurveZeros
ZERO RATE FREQUENCY: FrequencyTypes.SEMI_ANNUAL
DATES: ZERO RATES
31-JUL-2020:   0.01000000
01-JAN-2027:   0.02000000

OBJECT TYPE: DiscountCurve
VALUE DATE: 31-JUL-2020
    DATES      TIMES(YRS) DISC FACTORS
 31-JUL-2020:     0.000000  1.00000000
 01-JAN-2027:     6.513889  0.87841977
INTERPOLATION TYPE: LINEAR_ZERO_RATES
TIME DAY COUNT TYPE: ACT_360



Dirty price is the clean price plus accrued interest

In [13]:
print("Dirty Price = %12.7f"
      % bond.dirty_price_from_discount_curve(settle_dt, zero_curve))

Dirty Price =  111.6012254


In [14]:
print("Clean Price = %12.7f"
      % bond.clean_price_from_discount_curve(settle_dt, zero_curve))

Clean Price =  111.0317809


In [15]:
bond.print_payments(settle_dt)

Coupon Date 	 Payment Date 	         Status          Amount
08-NOV-2020 	             	     SETTLEMENT 
28-MAR-2021 	 29-MAR-2021 	   HOLIDAY ROLL 	            2.50 
28-SEP-2021 	 28-SEP-2021 	      UNCHANGED 	            2.50 
28-MAR-2022 	 28-MAR-2022 	      UNCHANGED 	            2.50 
28-SEP-2022 	 28-SEP-2022 	      UNCHANGED 	            2.50 
28-MAR-2023 	 28-MAR-2023 	      UNCHANGED 	            2.50 
28-SEP-2023 	 28-SEP-2023 	      UNCHANGED 	            2.50 
28-MAR-2024 	 28-MAR-2024 	      UNCHANGED 	            2.50 
28-SEP-2024 	 30-SEP-2024 	       MATURITY 	          102.50 



Accrued interest is accrued from previous coupon date to settlement date

In [16]:
print("Previous coupon date is ", bond._pcd)

Previous coupon date is  28-SEP-2020


In [17]:
print("Settlement date is ", settle_dt)

Settlement date is  08-NOV-2020


The amount of accrued interest is 

In [18]:
print("Accrued = %12.5f" % bond.accrued_int)

Accrued =      0.00569


This is based on the following number of days of accrual

In [19]:
print("Accrued Days = %d" % bond.accrued_days)

Accrued Days = 41


Copyright (c) 2020 Dominic O'Kane